### Ingesting Drivers .csv file

In [0]:
%run ../00-common/01-environment-config

In [0]:
%run ../00-common/02-bronze_helper

In [0]:
source_file = f"{landing_forlder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"


### Adding Ingestion data

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

drivers_schema = StructType([
    StructField("driverId", StringType(), True),
    StructField("name", StructType([
        StructField("givenName", StringType(), True),
        StructField("familyName", StringType(), True)
    ]), True),
    StructField("dateOfBirth", DateType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

drivers_df = (
    spark.read.format('json')
    .schema(drivers_schema)
    .option('mode', 'FAILFAST')
    .load(source_file)
    .select("*", "_metadata")
)


In [0]:
display(drivers_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8753923351933805>, line 1
----> 1 display(drivers_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"
     95       ) from e
---> 96 if df.isStreaming:
     97     self.cf_helper.display_streaming_dataframe(df, config

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

### Creating Delta table

In [0]:
(drivers_final_df
 .write
 .mode("overwrite")
 .format('delta')
 .saveAsTable(table_name)
)
display(spark.read.table(table_name))